# 05 - Selected citation source trace

This notebook audits the data behind Figure 9 and the related Appendix C mini bar check. It rebuilds the plotted source conference citation counts from the reference author edge table and writes a paper level trace for every counted edge.

## 1. Setup

In [7]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )


import json
import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

start = Path.cwd().resolve()
project_candidate = None
for candidate in [start, *start.parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        project_candidate = candidate
        break
    child_matches = sorted(
        child for child in candidate.iterdir()
        if child.is_dir() and (child / "config" / "project_config.yaml").exists()
    )
    if len(child_matches) == 1:
        project_candidate = child_matches[0]
        break

if project_candidate is None:
    raise FileNotFoundError("Could not find config/project_config.yaml")

if str(project_candidate) not in sys.path:
    sys.path.insert(0, str(project_candidate))

import pandas as pd
from IPython.display import display, Markdown

from author_matching import normalize_name, short_openalex_id
from project_setup import ensure_dirs, setup_project

setup = setup_project(project_candidate)
PROJECT = setup.project_folder

STEP_2_PREPARED = PROJECT / "step_2_data" / "prepared" / "all_papers"
STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
ensure_dirs(STEP_4_SUMMARY)

REF_AUTHORS_PATH = STEP_2_PREPARED / "all_ref_authors_exploded.parquet"
PAPERS_PATH = STEP_2_PREPARED / "all_papers_filtered.parquet"
CITATION_KEYS_PATH = STEP_3_PREPARED / "panel_citation_keys.parquet"
DISTRIBUTION_ROWS_PATH = STEP_4_SUMMARY / "selected_pc_source_distribution_rows.csv"
TRACE_OUT = STEP_4_SUMMARY / "selected_pc_source_share_citation_trace.csv"
RECONCILIATION_OUT = STEP_4_SUMMARY / "selected_count_check.csv"

print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Overwrite artifacts: {setup.overwrite_artifacts}")


Project folder: .
Run mode: fast
Overwrite artifacts: True


## 2. Helper functions

In [8]:

def source_group(conference):
    if conference in {"OOPSLA", "OOPSLA1", "OOPSLA2"}:
        return "OOPSLA"
    return conference


def join_unique(values):
    clean = sorted({str(v) for v in values if pd.notna(v) and str(v).strip()})
    return "; ".join(clean)


def as_short_openalex_id(value):
    if pd.isna(value):
        return pd.NA
    short = short_openalex_id(value)
    return short if short else pd.NA


def iter_authorships(authorships):
    if hasattr(authorships, "tolist"):
        return authorships.tolist()
    if isinstance(authorships, (list, tuple)):
        return list(authorships)
    return []


def paper_author_names(authorships):
    return "; ".join(
        author.get("author_name") or author.get("display_name") or ""
        for author in iter_authorships(authorships)
        if isinstance(author, dict)
        and (author.get("author_name") or author.get("display_name"))
    )


def paper_author_ids(authorships):
    return {
        author.get("author_id").rstrip("/").split("/")[-1]
        for author in iter_authorships(authorships)
        if isinstance(author, dict) and author.get("author_id")
    }


## 3. Read plotted cells and citation keys

In [9]:

distribution_rows = pd.read_csv(DISTRIBUTION_ROWS_PATH)
distribution_rows = distribution_rows.rename(columns={"conference": "selected_conference"})
distribution_rows["cell_id"] = range(len(distribution_rows))
selected_ids = sorted(distribution_rows["researcher_id"].unique())

citation_keys = pd.read_parquet(CITATION_KEYS_PATH)
selected_keys = citation_keys.loc[citation_keys["researcher_id"].isin(selected_ids)].copy()

ref_authors = pd.read_parquet(REF_AUTHORS_PATH)
refs = ref_authors.rename(
    columns={"issue": "source_conference", "conference_year": "year"}
).copy()
refs["source_group"] = refs["source_conference"].map(source_group)
refs["ref_author_id"] = refs["ref_author_id"].map(as_short_openalex_id)
refs["ref_author_name_norm"] = refs["ref_author_name"].map(normalize_name)
refs["event_row_id"] = range(len(refs))

print(f"Selected figure cells: {len(distribution_rows):,}")
print(f"Selected researchers: {len(selected_ids):,}")
print(f"Selected citation keys: {len(selected_keys):,}")
display(distribution_rows.head())


Selected figure cells: 200
Selected researchers: 8
Selected citation keys: 8


,plot_order,researcher_id,name,selected_conference,plot_conference,pc_year,selected_event_label,event_time,year,source_group,citation_count,total_citations,source_share,source_share_pct,pc_status_for_source,cell_id
0,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,ICFP,1.0,17.0,0.058824,5.882353,0.0,0
1,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,POPL,7.0,17.0,0.411765,41.176471,0.0,1
2,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,OOPSLA,3.0,17.0,0.176471,17.647059,0.0,2
3,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,PLDI,6.0,17.0,0.352941,35.294118,1.0,3
4,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-1,2023,ICFP,3.0,12.0,0.250000,25.000000,0.0,4


## 4. Rebuild unique citation edges

The panel counts one unique `(researcher, citing paper, cited work)` edge. This cell rebuilds those edges using the same OpenAlex author ID and accepted name key matching logic.

In [10]:

id_keys = selected_keys.loc[
    selected_keys["citation_key_type"].eq("openalex_author_id")
].copy()
name_keys = selected_keys.loc[
    ~selected_keys["citation_key_type"].eq("openalex_author_id")
].copy()

id_events = refs.merge(
    id_keys,
    left_on="ref_author_id",
    right_on="citation_key_value",
    how="inner",
)
name_events = refs.merge(
    name_keys,
    left_on="ref_author_name_norm",
    right_on="citation_key_value",
    how="inner",
)
raw_events = pd.concat([id_events, name_events], ignore_index=True)
raw_events = raw_events.loc[raw_events["researcher_id"].isin(selected_ids)].copy()

edge_cols = [
    "researcher_id",
    "source_conference",
    "year",
    "source_group",
    "work_id",
    "referenced_work_id",
]


def collapse_event_group(group):
    return pd.Series(
        {
            "ref_author_name": join_unique(group["ref_author_name"]),
            "ref_author_id": join_unique(group["ref_author_id"]),
            "ref_orcid": join_unique(group["ref_orcid"]),
            "cited_publication_year": pd.to_numeric(
                group["cited_publication_year"], errors="coerce"
            ).min(),
            "author_position": join_unique(group["author_position"]),
            "is_self_citation": bool(group["is_self_citation"].astype(bool).any()),
            "citation_key_type": join_unique(group["citation_key_type"]),
            "citation_key_value": join_unique(group["citation_key_value"]),
            "matched_reference_author_rows": int(len(group)),
        }
    )


citation_edges = (
    raw_events.groupby(edge_cols, group_keys=False, dropna=False)
    .apply(collapse_event_group)
    .reset_index()
)

print(f"Raw matched reference-author rows: {len(raw_events):,}")
print(f"Unique researcher/citing-paper/cited-work edges: {len(citation_edges):,}")
display(citation_edges.head())


Raw matched reference-author rows: 1,047
Unique researcher/citing-paper/cited-work edges: 1,045


,researcher_id,source_conference,year,source_group,work_id,referenced_work_id,ref_author_name,ref_author_id,ref_orcid,cited_publication_year,author_position,is_self_citation,citation_key_type,citation_key_value,matched_reference_author_rows
0,brandonlucia,OOPSLA,2017,OOPSLA,W2761242776,W2115721786,Brandon Lucia,A5076249757,0000-0003-4130-1099,2015.0,first,True,openalex_author_id,A5076249757,1
1,brandonlucia,OOPSLA,2017,OOPSLA,W2761242776,W2122912905,Brandon Lucia,A5076249757,0000-0003-4130-1099,2014.0,last,True,openalex_author_id,A5076249757,1
2,brandonlucia,OOPSLA,2017,OOPSLA,W2761242776,W2324544651,Brandon Lucia,A5076249757,0000-0003-4130-1099,2016.0,middle,True,openalex_author_id,A5076249757,1
3,brandonlucia,OOPSLA,2017,OOPSLA,W2761242776,W2537482850,Brandon Lucia,A5076249757,0000-0003-4130-1099,2016.0,last,True,openalex_author_id,A5076249757,1
4,brandonlucia,OOPSLA,2017,OOPSLA,W2761380876,W2134440791,Brandon Lucia,A5076249757,0000-0003-4130-1099,2009.0,middle,False,openalex_author_id,A5076249757,1


## 5. Add paper metadata

In [11]:

papers_full = pd.read_parquet(PAPERS_PATH)
papers = papers_full[
    ["work_id", "doi", "title", "conference", "conference_year", "issue", "authorships"]
].copy()
papers["citing_authors"] = papers["authorships"].map(paper_author_names)
paper_author_id_map = dict(
    zip(papers["work_id"], papers["authorships"].map(paper_author_ids))
)
paper_lookup = papers.rename(
    columns={
        "work_id": "citing_work_id",
        "doi": "citing_doi",
        "title": "citing_title",
        "conference": "citing_conference",
        "conference_year": "citing_year",
        "issue": "citing_issue",
    }
)[
    [
        "citing_work_id",
        "citing_doi",
        "citing_title",
        "citing_conference",
        "citing_year",
        "citing_issue",
        "citing_authors",
    ]
]

citation_edges = citation_edges.rename(columns={"work_id": "citing_work_id"})
citation_edges = citation_edges.merge(
    paper_lookup,
    on="citing_work_id",
    how="left",
    validate="many_to_one",
)

cache_dirs = [
    PROJECT / "step_2_data" / "raw" / "openalex_cache" / "2026-06-11" / "works",
    PROJECT / "step_2_data" / "raw" / "openalex_cache" / "works",
]
work_cache = {}


def load_work(work_id):
    if pd.isna(work_id):
        return {}
    if work_id in work_cache:
        return work_cache[work_id]
    for cache_dir in cache_dirs:
        path = cache_dir / f"{work_id}.json"
        if path.exists():
            work_cache[work_id] = json.loads(path.read_text())
            return work_cache[work_id]
    work_cache[work_id] = {}
    return work_cache[work_id]


citation_edges["cited_work_title"] = citation_edges["referenced_work_id"].map(
    lambda work_id: load_work(work_id).get("title")
    or load_work(work_id).get("display_name")
    or pd.NA
)
citation_edges["cited_work_doi"] = citation_edges["referenced_work_id"].map(
    lambda work_id: load_work(work_id).get("doi") or pd.NA
)
citation_edges["cited_work_age_years"] = (
    citation_edges["year"] - citation_edges["cited_publication_year"]
)

cited_author_groups = (
    ref_authors.assign(
        ref_author_id_short=ref_authors["ref_author_id"].map(as_short_openalex_id)
    )
    .groupby(["work_id", "referenced_work_id"], dropna=False)
    .agg(
        cited_author_ids=(
            "ref_author_id_short",
            lambda x: {str(v) for v in x if pd.notna(v) and str(v).strip()},
        ),
        cited_author_names=(
            "ref_author_name",
            lambda x: sorted({str(v) for v in x if pd.notna(v) and str(v).strip()}),
        ),
    )
    .reset_index()
    .rename(columns={"work_id": "citing_work_id"})
)
citation_edges = citation_edges.merge(
    cited_author_groups,
    on=["citing_work_id", "referenced_work_id"],
    how="left",
    validate="many_to_one",
)


def overlap_info(row):
    citing_ids = paper_author_id_map.get(row["citing_work_id"], set())
    cited_ids = row["cited_author_ids"] if isinstance(row["cited_author_ids"], set) else set()
    overlap_ids = sorted(citing_ids & cited_ids)
    cited_names = row["cited_author_names"] if isinstance(row["cited_author_names"], list) else []
    return pd.Series(
        {
            "has_citing_author_on_cited_work": bool(overlap_ids),
            "citing_author_overlap_ids": "; ".join(overlap_ids),
            "citing_author_overlap_names": "; ".join(cited_names) if overlap_ids else "",
        }
    )


citation_edges = pd.concat(
    [citation_edges, citation_edges.apply(overlap_info, axis=1)], axis=1
)


## 6. Check

In [12]:
cell_cols = [
    "cell_id",
    "plot_order",
    "researcher_id",
    "name",
    "selected_conference",
    "plot_conference",
    "pc_year",
    "selected_event_label",
    "event_time",
    "year",
    "source_group",
    "citation_count",
    "total_citations",
    "source_share",
    "source_share_pct",
    "pc_status_for_source",
]
trace = distribution_rows[cell_cols].merge(
    citation_edges,
    on=["researcher_id", "year", "source_group"],
    how="left",
)
trace_detail = trace.loc[trace["citing_work_id"].notna()].copy()
trace_detail["is_related_source_conference"] = trace_detail["source_group"].eq(
    trace_detail["selected_conference"].map(source_group)
)
trace_detail["served_on_source_pc_this_year"] = (
    trace_detail["pc_status_for_source"].fillna(0).astype(int)
)
trace_detail["edge_counted_in_panel"] = 1

trace_cols = [
    "plot_order",
    "cell_id",
    "researcher_id",
    "name",
    "selected_conference",
    "plot_conference",
    "pc_year",
    "selected_event_label",
    "event_time",
    "year",
    "source_group",
    "source_conference",
    "is_related_source_conference",
    "served_on_source_pc_this_year",
    "citation_count",
    "total_citations",
    "source_share_pct",
    "citing_work_id",
    "citing_title",
    "citing_authors",
    "citing_doi",
    "referenced_work_id",
    "cited_work_title",
    "cited_work_doi",
    "ref_author_name",
    "ref_author_id",
    "ref_orcid",
    "cited_publication_year",
    "cited_work_age_years",
    "citation_key_type",
    "citation_key_value",
    "author_position",
    "is_self_citation",
    "has_citing_author_on_cited_work",
    "citing_author_overlap_ids",
    "citing_author_overlap_names",
    "matched_reference_author_rows",
    "edge_counted_in_panel",
]
trace_detail = trace_detail[trace_cols].sort_values(
    [
        "plot_order",
        "event_time",
        "source_group",
        "citing_title",
        "cited_work_title",
        "referenced_work_id",
    ],
    na_position="last",
).reset_index(drop=True)

rebuilt_counts = (
    trace_detail.groupby("cell_id", dropna=False)
    .size()
    .rename("rebuilt_citation_count_raw")
    .reset_index()
)
reconciliation = distribution_rows.merge(rebuilt_counts, on="cell_id", how="left")
reconciliation["rebuilt_citation_count_raw"] = (
    reconciliation["rebuilt_citation_count_raw"].fillna(0).astype(int)
)
reconciliation["panel_cell_available"] = reconciliation["citation_count"].notna()
reconciliation["plotted_citation_count"] = reconciliation["citation_count"]
reconciliation["rebuilt_citation_count"] = reconciliation[
    "rebuilt_citation_count_raw"
].astype("float")
reconciliation.loc[
    ~reconciliation["panel_cell_available"], "rebuilt_citation_count"
] = pd.NA
reconciliation["count_matches_trace"] = pd.NA
available = reconciliation["panel_cell_available"]
reconciliation.loc[available, "count_matches_trace"] = reconciliation.loc[
    available, "rebuilt_citation_count_raw"
].eq(reconciliation.loc[available, "citation_count"].astype(int))

source_totals = (
    reconciliation.groupby(
        ["plot_order", "researcher_id", "pc_year", "event_time", "year"],
        dropna=False,
    )
    .agg(
        rebuilt_total_citations_raw=("rebuilt_citation_count_raw", "sum"),
        plotted_total_citations=("total_citations", "first"),
    )
    .reset_index()
)
reconciliation = reconciliation.merge(
    source_totals,
    on=["plot_order", "researcher_id", "pc_year", "event_time", "year"],
    how="left",
    validate="many_to_one",
)
reconciliation["total_matches_trace"] = pd.NA
total_available = reconciliation["plotted_total_citations"].notna()
reconciliation.loc[total_available, "total_matches_trace"] = reconciliation.loc[
    total_available, "rebuilt_total_citations_raw"
].astype(float).eq(reconciliation.loc[total_available, "plotted_total_citations"].astype(float))

reconciliation = reconciliation[
    [
        "plot_order",
        "cell_id",
        "researcher_id",
        "name",
        "selected_conference",
        "plot_conference",
        "pc_year",
        "selected_event_label",
        "event_time",
        "year",
        "source_group",
        "panel_cell_available",
        "plotted_citation_count",
        "rebuilt_citation_count",
        "count_matches_trace",
        "total_citations",
        "rebuilt_total_citations_raw",
        "total_matches_trace",
        "source_share_pct",
        "pc_status_for_source",
    ]
].sort_values(["plot_order", "event_time", "source_group"]).reset_index(drop=True)

trace_detail.to_csv(TRACE_OUT, index=False)
reconciliation.to_csv(RECONCILIATION_OUT, index=False)

available_cells = reconciliation["panel_cell_available"].fillna(False)
cell_mismatches = int(
    (~reconciliation.loc[available_cells, "count_matches_trace"].astype(bool)).sum()
)
available_totals = reconciliation.drop_duplicates(["plot_order", "event_time"])
available_totals = available_totals.loc[available_totals["total_citations"].notna()]
total_mismatches = int((~available_totals["total_matches_trace"].astype(bool)).sum())
missing_cells = int((~reconciliation["panel_cell_available"]).sum())
print(f"Trace rows: {len(trace_detail):,}")
print(f"Reconciliation rows: {len(reconciliation):,}")
print(f"Available plotted cells: {int(available_cells.sum()):,}")
print(f"Unavailable plotted cells: {missing_cells:,}")
print(f"Cell mismatches: {cell_mismatches:,}")
print(f"Total mismatches: {total_mismatches:,}")
assert cell_mismatches == 0
assert total_mismatches == 0

display(Markdown(f"`{TRACE_OUT.relative_to(PROJECT)}`"))
display(Markdown(f"`{RECONCILIATION_OUT.relative_to(PROJECT)}`"))
display(reconciliation.head(12))


Trace rows: 885
Reconciliation rows: 200
Available plotted cells: 180
Unavailable plotted cells: 20
Cell mismatches: 0
Total mismatches: 0


`step_4_artifacts/summary_tables/selected_pc_source_share_citation_trace.csv`

`step_4_artifacts/summary_tables/selected_count_check.csv`

,plot_order,cell_id,researcher_id,name,selected_conference,plot_conference,pc_year,selected_event_label,event_time,year,source_group,panel_cell_available,plotted_citation_count,rebuilt_citation_count,count_matches_trace,total_citations,rebuilt_total_citations_raw,total_matches_trace,source_share_pct,pc_status_for_source
0,1,0,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,ICFP,True,1.0,1.0,True,17.0,17,True,5.882353,0.0
1,1,2,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,OOPSLA,True,3.0,3.0,True,17.0,17,True,17.647059,0.0
2,1,3,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,PLDI,True,6.0,6.0,True,17.0,17,True,35.294118,1.0
3,1,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-2,2022,POPL,True,7.0,7.0,True,17.0,17,True,41.176471,0.0
4,1,4,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-1,2023,ICFP,True,3.0,3.0,True,12.0,12,True,25.000000,0.0
5,1,6,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-1,2023,OOPSLA,True,2.0,2.0,True,12.0,12,True,16.666667,0.0
6,1,7,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-1,2023,PLDI,True,3.0,3.0,True,12.0,12,True,25.000000,0.0
7,1,5,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,-1,2023,POPL,True,4.0,4.0,True,12.0,12,True,33.333333,1.0
8,1,8,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,0,2024,ICFP,True,1.0,1.0,True,20.0,20,True,5.000000,1.0
9,1,10,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,0,2024,OOPSLA,True,2.0,2.0,True,20.0,20,True,10.000000,0.0
